In [1]:
import pandas as pd
import re

In [21]:
arabic_articles = pd.read_csv('arabic_articles_preprocessed.csv')
ara_sum = pd.read_csv('cleaned_arasum_final.csv')
egy_sum = pd.read_csv("cleaned_egyptian_final.csv")
sum_arabic = pd.read_csv("cleaned_sumarabic_final.csv")

In [23]:
print(f"ara_sum Columns: {ara_sum.columns.to_list()}")
print(f"egy_sum Columns: {egy_sum.columns.to_list()}")
print(f"sum_arabic Columns: {sum_arabic.columns.to_list()}")
print(f"arabic_articles Columns: {arabic_articles.columns.to_list()}")


ara_sum Columns: ['article', 'summary']
egy_sum Columns: ['text', 'summarized_text', 'source_topics', 'id']
sum_arabic Columns: ['text', 'headline', 'section', 'published', 'url', 'split']
arabic_articles Columns: ['id', 'type', 'text', 'clean_text', 'word_count', 'clean_word_count', 'arabic_ratio']


# Applying Cleanup (English Norm.)

In [33]:
def final_unified_cleanup(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace("<start>", "").replace("<end>", "")

    # optional: lowercase English only
    def lower_english(match):
        return match.group(0).lower()

    text = re.sub(r'[A-Za-z]+', lower_english, text)
    return text

In [27]:
def inspect_remaining_english(df, text_col, summary_col, n_samples=20):
    english_pattern = re.compile(r'[A-Za-z]')

    def extract_english_words(text):
        if not isinstance(text, str):
            return []
        return re.findall(r'[A-Za-z]+', text)

    filtered = df[
        df[text_col].fillna('').str.contains(english_pattern) |
        df[summary_col].fillna('').str.contains(english_pattern)
    ].copy()

    filtered[f'{text_col}_english_words'] = filtered[text_col].apply(extract_english_words)
    filtered[f'{summary_col}_english_words'] = filtered[summary_col].apply(extract_english_words)

    print(f"Rows with English in either '{text_col}' or '{summary_col}': {len(filtered)}")

    display(
        filtered[
            [text_col, summary_col, f'{text_col}_english_words', f'{summary_col}_english_words']
        ].head(n_samples)
    )

    return filtered

In [34]:
ara_sum['article'] = ara_sum['article'].apply(final_unified_cleanup)
ara_sum['summary'] = ara_sum['summary'].apply(final_unified_cleanup)
remaining_english = inspect_remaining_english(ara_sum, 'article', 'summary')

Rows with English in either 'article' or 'summary': 9901


,article,summary,article_english_words,summary_english_words
0,"حقق حزب ""البديل من اجل المانيا"" اليميني الشعبو...",تشير اخر استطلاعات الراي الالمانية الي تقدم ح...,"[ard, zdf]",[]
10,مرة اخري ينهار اتفاق الهدنة في سوريا ويلقي نفس...,منذ اربع سنوات، والصراع علي مدينة حلب يزداد ح...,"[dw, dw]",[]
12,"""اشرف"" شاب يبلغ من العمر 28 عاما وحاصل علي شها...","""حلمت بمستقبل افضل لكن الحرب والبطالة دفعتا ب...","[dw, dw, dw, dw]",[dw]
14,:dwسيد يونين، تعرضت قافلة مساعدات انسانية للقص...,الحق الاعتداء الجوي علي قافلة مساعدات انسانية...,[dw],[dw]
21,قبل سنتين وشهرين فر ضباط الشرطة، الذين كانوا ي...,"مثل حكم تنظيم ""الدولة الاسلامية"" لمدينة القيا...",[],[dw]
22,بشكل عام، يحق للمشتري اعادة البضاعة المشتراة ف...,يحمي القانون المستهلك والبائع في تجارة المفرد...,"[shopping, dw]",[]
30,عامان من المعاناة عاشتها ارنستين يولي بسبب تور...,يعتبر تورم الذراعين او الساقين من ابرز اعراض ...,"[dw, dw]",[]
35,يعد توماس شاف، المدرب السابق لفيردر بريمن، صاح...,ارسين وارسنال، لا يجمعهما التجانس اللغوي في ل...,[iffhs],[]
37,"فازت مزن حسن الناشطة النسوية ومؤسسة ""نظرة"" للد...","اثر فوزها بجائزة ""نوبل البديلة"" قالت الناشطة ...","[dw, dw]",[dw]
50,صحفيات وصحفيون من اكثر من 60 دولة هم مفتاح نجا...,موقع dw عربية في بون يبحث عن: متدربات/ متدربي...,"[dw, dw, dw, pdf, mb, bewerbung, dw, com, stel...",[dw]


In [35]:
egy_sum['text'] = egy_sum['text'].apply(final_unified_cleanup)
egy_sum['summarized_text'] = egy_sum['summarized_text'].apply(final_unified_cleanup)
remaining_english = inspect_remaining_english(egy_sum, 'text', 'summarized_text')

Rows with English in either 'text' or 'summarized_text': 268


,text,summarized_text,text_english_words,summarized_text_english_words
68,تطبيقات النانو تكنولوجي في تنقية المياه والهوا...,النانو تكنولوجي بيستخدم اغشية ومواد دقيقة جدا...,[nanofilters],[]
120,الهندسة الوراثية في الزراعة سمحت لينا نطور محا...,القطن المعدل وراثيا بجين ال bt بيقاوم دودة ال...,[bt],[bt]
125,تقنية الكريسبر-كاس9 (crispr-cas9) تعتبر من اقو...,الكريسبر-كاس9 هي اداة قوية ودقيقة لتعديل جينا...,"[crispr, cas]",[]
136,من اشهر طرق احداث الطفرات استخدام الاشعاع، زي ...,الاشعاع (زي جاما وسينية) طريقة شائعة لاحداث ا...,[dna],[dna]
137,المواد الكيميائية المطفرة زي ال ems (ايثيل ميث...,كيماويات زي ال ems بتستخدم لاحداث الطفرات، بت...,"[ems, dna]","[ems, dna]"
155,تداخل الحمض النووي الريبوزي (rnai) في النباتات...,rnai في النباتات عملية طبيعية بتنظم الجينات ع...,"[rnai, rna, mrna]","[rnai, rna]"
156,النباتات بتستخدم الية تداخل الحمض النووي الريب...,النباتات بتستخدم rnai عشان تدافع عن نفسها ضد ...,"[rnai, rna, rna]","[rnai, rna]"
157,علماء كتير بيشتغلوا علي استخدام تقنية rnai عشا...,تقنية rnai بتستخدم لتطوير نباتات مقاومة للحشر...,"[rnai, rna]","[rnai, rna]"
158,تداخل الحمض النووي الريبوزي (rnai) ليه تطبيقات...,rnai بيستخدم لتحسين خصائص المحاصيل، زي تاخير ...,[rnai],[rnai]
159,بعيدا عن تطبيقاتها في مقاومة الامراض والافات، ...,rnai مش بس للدفاع، دي الية اساسية في النباتات...,[rnai],[rnai]


In [36]:
sum_arabic['text'] = sum_arabic['text'].apply(final_unified_cleanup)
sum_arabic['headline'] = sum_arabic['headline'].apply(final_unified_cleanup)
remaining_english = inspect_remaining_english(sum_arabic, 'text', 'headline')

Rows with English in either 'text' or 'headline': 2363


,text,headline,text_english_words,headline_english_words
180,اقامت جامعة عجمان للعلوم والتكنولوجيا يوما الم...,يوم الماني في جامعة عجمان,[daad],[]
607,حصلت شركة مستحضرات تنظيف وتبييض الثياب «كلوركس...,«كلوركس» تحصد 3 ذهبيات من جوائز «جيماس»,[effies],[]
748,سيقوم «ايرمايلز»، برنامج مكافاة المستهلك علي ا...,«ايرمايلز» يحقق حلم طالب لاحتراف «الرجبي»,[jess],[]
865,افاد مدير عام هيئة الصحة في دبي، قاضي المروشد،...,مستشفيات دبي تحصل علي «الاعتماد الدولي» لخدما...,[jci],[]
1023,حصلت شركة العقارات «فالكن ستي اوف ووندرز»، علي...,جائزة ل«فالكن ستي» لمشاركتها ب«معرض العقارات ...,[ips],[]
1305,حصل مركز التسوق «لامسي بلازا»، علي جائزة retai...,«لامسي بلازا» يحصد جائزة «retailme» لمكافاة ا...,[retailme],[retailme]
1531,اطلقت هيئة المعرفة والتنمية البشرية في دبي، ام...,لجنة «رقابة وتفتيش» علي جامعات في دبي,[uqaib],[]
1658,اعلنت vlcc شركة الرعاية الصحية الجمالية والوقا...,vlcc تواصل حملة التوعية احتفالا بالامومة,[vlcc],[vlcc]
1816,برعاية مجلس دبي الثقافي، اقيم اخيرا حفل توقيع ...,الريس يوقع «الوان حياتي» في مهرجان الخور,[xva],[]
1924,كشف تقرير لاتحاد المستهلكين العضويين الاميركي«...,24 سلعة اميركية مسرطنة تتداول في الاسواق,[oca],[]


In [37]:
arabic_articles['text'] = arabic_articles['text'].apply(final_unified_cleanup)

# **Tokenization and Dataset merging**

See the README file for more details

# Unify Column Names

In [38]:
# AraSum
df_ara = ara_sum.copy()
df_ara = df_ara.rename(columns={
    "article": "source_text",
    "summary": "target_text"
})
df_ara["dataset_name"] = "arasum"

# Egyptian
df_egy = egy_sum.copy()
df_egy = df_egy.rename(columns={
    "text": "source_text",
    "summarized_text": "target_text"
})
df_egy["dataset_name"] = "egyptian"

# SumArabic
df_sum = sum_arabic.copy()
df_sum = df_sum.rename(columns={
    "text": "source_text",
    "headline": "target_text"
})
df_sum["dataset_name"] = "sumarabic"

# Keep only the columns we actually need
df_ara = df_ara[["source_text", "target_text", "dataset_name"]].copy()
df_egy = df_egy[["source_text", "target_text", "dataset_name"]].copy()
df_sum = df_sum[["source_text", "target_text", "dataset_name", "split"]].copy()

# Final safety cleanup
for df_ in [df_ara, df_egy, df_sum]:
    df_.dropna(subset=["source_text", "target_text"], inplace=True)
    df_.drop_duplicates(subset=["source_text", "target_text"], inplace=True)
    df_.reset_index(drop=True, inplace=True)

print("AraSum:", df_ara.shape)
print("Egyptian:", df_egy.shape)
print("SumArabic:", df_sum.shape)

AraSum: (49582, 3)
Egyptian: (3689, 3)
SumArabic: (84602, 4)


# Splitting

In [39]:
from sklearn.model_selection import train_test_split

def make_splits(df, train_size=0.8, valid_size=0.1, test_size=0.1, seed=42):
    assert abs(train_size + valid_size + test_size - 1.0) < 1e-9

    train_df, temp_df = train_test_split(df, test_size=(1 - train_size), random_state=seed, shuffle=True)
    valid_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (valid_size + test_size),
        random_state=seed,
        shuffle=True
    )

    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    train_df["split"] = "train"
    valid_df["split"] = "valid"
    test_df["split"] = "test"

    return train_df, valid_df, test_df

In [40]:
# Create splits for AraSum and Egyptian
ara_train, ara_valid, ara_test = make_splits(df_ara)
egy_train, egy_valid, egy_test = make_splits(df_egy)

# SumArabic already has split
sum_train = df_sum[df_sum["split"] == "train"].copy()
sum_valid = df_sum[df_sum["split"] == "valid"].copy()
sum_test  = df_sum[df_sum["split"] == "test"].copy()
sum_ood   = df_sum[df_sum["split"] == "ood"].copy()

# Creating a Dataset for each experiment  

In [41]:
# Experiment A: AraSum + Egyptian
expA_train = pd.concat([ara_train, egy_train], ignore_index=True)
expA_valid = pd.concat([ara_valid, egy_valid], ignore_index=True)
expA_test  = pd.concat([ara_test, egy_test], ignore_index=True)

# Experiment B: AraSum + Egyptian + SumArabic
expB_train = pd.concat([ara_train, egy_train, sum_train], ignore_index=True)
expB_valid = pd.concat([ara_valid, egy_valid, sum_valid], ignore_index=True)
expB_test  = pd.concat([ara_test, egy_test, sum_test], ignore_index=True)

# Experiment C:
expC_stage1_train = sum_train.copy()                       # stage 1: SumArabic
expC_stage2_train = pd.concat([ara_train, egy_train], ignore_index=True)  # stage 2: AraSum + Egyptian
expC_valid = pd.concat([ara_valid, egy_valid, sum_valid], ignore_index=True)
expC_test  = pd.concat([ara_test, egy_test, sum_test], ignore_index=True)

print("Experiment A:")
print("train:", expA_train.shape, "valid:", expA_valid.shape, "test:", expA_test.shape)

print("Experiment B:")
print("train:", expB_train.shape, "valid:", expB_valid.shape, "test:", expB_test.shape)

print("Experiment C:")
print("stage1 train:", expC_stage1_train.shape)
print("stage2 train:", expC_stage2_train.shape)
print("valid:", expC_valid.shape, "test:", expC_test.shape)
print("ood (optional SumArabic eval):", sum_ood.shape)

Experiment A:
train: (42616, 4) valid: (5327, 4) test: (5328, 4)
Experiment B:
train: (118284, 4) valid: (9444, 4) test: (9495, 4)
Experiment C:
stage1 train: (75668, 4)
stage2 train: (42616, 4)
valid: (9444, 4) test: (9495, 4)
ood (optional SumArabic eval): (650, 4)


# Tokenization

In [42]:
import sentencepiece as spm
from pathlib import Path

TOKENIZER_DIR = Path("tokenizers")
TOKENIZER_DIR.mkdir(exist_ok=True)

In [43]:
def write_sp_corpus(df, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        for src, tgt in zip(df["source_text"], df["target_text"]):
            f.write(src.strip() + "\n")
            f.write(tgt.strip() + "\n")

def train_sentencepiece(
    input_path,
    model_prefix,
    vocab_size=16000,
    model_type="unigram"
):
    spm.SentencePieceTrainer.Train(
        input=input_path,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type=model_type,
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<start>",
        eos_piece="<end>",
        normalization_rule_name="identity"
    )

In [44]:
corpus_A = TOKENIZER_DIR / "corpus_expA.txt"
corpus_BC = TOKENIZER_DIR / "corpus_expBC.txt"

write_sp_corpus(expA_train, corpus_A)
write_sp_corpus(expB_train, corpus_BC)

In [45]:
train_sentencepiece(
    input_path=str(corpus_A),
    model_prefix=str(TOKENIZER_DIR / "sp_expA"),
    vocab_size=16000,
    model_type="unigram"
)

train_sentencepiece(
    input_path=str(corpus_BC),
    model_prefix=str(TOKENIZER_DIR / "sp_expBC"),
    vocab_size=16000,
    model_type="unigram"
)

In [46]:
print("Done training SentencePiece tokenizers.")
print("Experiment A tokenizer:", TOKENIZER_DIR / "sp_expA.model")
print("Experiment B/C tokenizer:", TOKENIZER_DIR / "sp_expBC.model")

Done training SentencePiece tokenizers.
Experiment A tokenizer: tokenizers/sp_expA.model
Experiment B/C tokenizer: tokenizers/sp_expBC.model


# Testing tokenizers

In [47]:
spA = spm.SentencePieceProcessor()
spA.load(str(TOKENIZER_DIR / "sp_expA.model"))

spBC = spm.SentencePieceProcessor()
spBC.load(str(TOKENIZER_DIR / "sp_expBC.model"))

sample_text = expA_train["source_text"].iloc[0]
sample_target = expA_train["target_text"].iloc[0]

print("Sample source:")
print(sample_text)

print("A tokenizer pieces:")
print(spA.encode(sample_text, out_type=str)[:40])

print("A tokenizer ids:")
print(spA.encode(sample_text, out_type=int)[:40])

print("Sample target:")
print(sample_target)

print("B/C tokenizer pieces:")
print(spBC.encode(sample_target, out_type=str)[:40])

Sample source:
dw تركيا بلد يعج بالمشاكل واردوغان حاكم مستبد والاكراد يتعرضون للقمع هل هذه هي نظرة وسائل الاعلام الالمانية لتركيا؟ تشانا افير: جزء كبير من وسائل الاعلام يقوم بتغطية اعلامية يشوبها النقص. هذا له بشكل عام شرعيته، لكن غالبا ما ينقص التمييز. فلماذا؟ لانه قلما يتم النظر الي الكواليس الخلفية، ولاسيما المشاكل السياسية الداخلية لا يتم الكشف عنها علي الوجه الصحيح. ما هي القضايا السياسية والاجتماعية التي تغطيها وسائل الاعلام الالمانية بكفاءة وتميز؟ القليل من وسائل الاعلام الالمانية يقدم تقارير عن الدور الحقيقي لحزب العمال الكردستاني الذي تم تصنيفه في المانيا وايضا الاتحاد الاوروبي كمنظمة ارهابية. اذن ليس فقط الالمان من اصل تركي او السياسة التركية الرسمية تسمي حزب العمال الكردستاني منظمة ارهابية. هناك ايضا مؤسسات اعلامية تتحدث مثلا عن هذا، وبعض وسائل الاعلام المكتوبة تقوم بتغطية جد محايدة. مثال اخر: ليس جميع الاتراك في المانيا ينتخبون حزب العدالة والتنمية. هناك حركة معارضة واسعة في تركيا وحتي بين الاتراك المقيمين في الخارج. وهذا لا يتم طرحه دوما في المانيا من الناحية الاعلامية بشك

In [48]:
print(spA.id_to_piece(0), spA.id_to_piece(1), spA.id_to_piece(2), spA.id_to_piece(3))
print(spBC.id_to_piece(0), spBC.id_to_piece(1), spBC.id_to_piece(2), spBC.id_to_piece(3))

<pad> <unk> <start> <end>
<pad> <unk> <start> <end>


# Download Datasets

In [50]:
expA_train.to_csv("expA_train.csv")
expA_valid.to_csv("expA_val.csv")
expA_test.to_csv("expA_test.csv")

expB_train.to_csv("expB_train.csv")
expB_valid.to_csv("expB_val.csv")
expB_test.to_csv("expB_test.csv")

expC_stage1_train.to_csv("expC_stage1_train.csv")
expC_stage2_train.to_csv("expC_stage2_train.csv")
expC_valid.to_csv("expC_val.csv")
expC_test.to_csv("expC_test.csv")
sum_ood.to_csv("sum_ood.csv")